# LO2 - LangChain 핵심 구성요소

## 이 챕터에서 익히는 것

- `init_chat_model`로 벤더 독립적인 ChatModel을 초기화하고 메시지(System/Human/AI)로 호출하기
- 멀티턴 대화 구성, 호출 파라미터(temperature 등) 조절, 토큰 스트리밍
- 같은 코드로 다른 벤더(Gemini) 전환하기
- `ChatPromptTemplate`과 LCEL 파이프(`prompt | model`)로 재사용 가능한 체인 만들기
- `with_structured_output`으로 응답을 Pydantic 스키마에 맞춰 구조화해 받기

각 예제는 **무엇을·왜 → 코드 → 체크포인트** 순서입니다. 위에서 아래로 순서대로 실행하면 동작합니다.

## 환경 준비

### 패키지 설치 (지정 버전)

LangChain 1.x는 API 변동이 잦으므로 지정 버전으로 설치합니다.

In [ ]:
!pip install -U "langchain==1.3.4" "langgraph==1.2.4" langchain-openai langchain-google-genai pydantic

### API 키 설정 — Colab과 로컬 둘 다

키는 **코드에 직접 적지 않고** 환경변수로 주입합니다.

- **Colab**: 좌측 열쇠 아이콘(시크릿)에 `OPENAI_API_KEY`를 등록한 뒤 `userdata`로 불러옵니다.
- **로컬/IDE**: 셸에서 `export OPENAI_API_KEY="sk-..."` 한 뒤 `os.getenv`로 읽습니다.

아래 셀은 Colab이면 `userdata`, 로컬이면 환경변수를 자동으로 인식합니다.

In [ ]:
import os

try:
    # Colab: 좌측 열쇠 아이콘에 등록한 시크릿을 환경변수로 주입합니다.
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    # Gemini 전환 예제를 실행하려면 GOOGLE_API_KEY도 등록해 두십시오.
    try:
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except Exception:
        pass  # Gemini를 안 쓰면 없어도 됩니다
    print("Colab 환경: userdata에서 키를 불러왔습니다.")
except ImportError:
    # 로컬/IDE: 셸 환경변수(export ...) 또는 .env에서 미리 설정된 값을 그대로 사용합니다.
    print("로컬 환경: os.getenv로 환경변수를 읽습니다.")

# 어느 환경이든 아래처럼 환경변수에서 읽어 씁니다.
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 설정되지 않았습니다."
print("OPENAI_API_KEY 확인됨")

### 모델 지정

모델은 `"벤더:모델명"` 문자열 하나로 지정합니다. 이 변수만 바꾸면 이후 모든 셀이 같은 모델을 씁니다.

> google-genai로 전환하려면 `MODEL = "google-genai:gemini-2.5-flash"`로 바꾸기만 하면 됩니다 (`GOOGLE_API_KEY` 필요).

In [ ]:
MODEL = "openai:gpt-5.4-mini"

> **체크포인트**: 설치 로그에 `langchain 1.3.4`가 보이고, 키 확인 셀이 오류 없이 `OPENAI_API_KEY 확인됨`을 출력하면 준비 완료입니다.

---
## 기본 예제

### ① 모델 초기화·invoke와 AIMessage 구조 관찰

`init_chat_model`로 모델 객체를 만들고, **메시지 리스트**를 `invoke`에 넘깁니다. 반환값은 단순 문자열이 아니라 `AIMessage` 객체라는 점을 확인합니다.

In [ ]:
from langchain.chat_models import init_chat_model
# v1 권장 경로. langchain_core.messages에서 가져와도 동일하게 동작합니다.
from langchain.messages import SystemMessage, HumanMessage, AIMessage

model = init_chat_model(MODEL)

# 시스템 메시지로 역할을 먼저 정하고, 사용자 메시지로 질문합니다.
resp = model.invoke([
    SystemMessage("너는 친절한 한국어 비서다."),  # 역할·규칙은 맨 앞에 한 번만
    HumanMessage("LCEL이 한 문장으로 뭐야?"),      # 사용자 입력
])

# 텍스트는 .content에 들어 있습니다.
print("[content]", resp.content)
print("[type]   ", type(resp))           # 예: <class 'langchain_core.messages.ai.AIMessage'>
print("[tool_calls]", resp.tool_calls)   # 예: []  (도구를 안 붙였으니 비어 있음, LO3에서 채워집니다)
print("[usage]  ", resp.usage_metadata)  # 예: {'input_tokens': 23, 'output_tokens': 41, ...}

> **체크포인트**: `resp.content`에 한국어 답변이 나오고, `type`이 `AIMessage`이면 성공입니다.

> **변형 포인트**: `SystemMessage`를 "초등학생도 알게 비유로 설명한다"로 바꿔 다시 호출하면 같은 질문이라도 톤이 달라집니다.

### ② System·Human·AI 멀티턴 메시지 구성

대화 기록을 메시지 리스트로 직접 쌓아 전달하면 모델이 맥락을 이어 받습니다. `AIMessage`는 "이전에 모델이 이렇게 답했다"는 과거 턴을 표현합니다.

In [ ]:
messages = [
    SystemMessage("너는 한 단어로만 답하는 비서다."),  # 형식 제약을 시스템에서 고정
    HumanMessage("대한민국의 수도는?"),
    AIMessage("서울"),                                  # 모델이 이미 이렇게 답했다고 가정한 과거 턴
    HumanMessage("그 도시의 인구는 대략 몇 명이야?"),    # "그 도시"는 위 맥락(서울)을 가리킵니다
]

resp = model.invoke(messages)
print("[answer]", resp.content)  # 예: 약 940만

# 후속 턴을 더 쌓으려면 응답을 리스트에 append하고 다시 invoke합니다.
messages.append(resp)
messages.append(HumanMessage("방금 말한 수치의 출처는 어디야?"))
print("[follow-up]", model.invoke(messages).content)

> **체크포인트**: "그 도시"가 서울로 이어지면 멀티턴 맥락이 정상 전달된 것입니다.

### ③ 호출 파라미터(temperature 등) 효과 비교

`temperature`는 답변의 무작위성을 조절합니다. 0에 가까울수록 결정적(매번 비슷), 높을수록 다양·창의적입니다. 파라미터는 모델을 만들 때 지정합니다.

In [ ]:
cold = init_chat_model(MODEL, temperature=0.0)  # 일관된 답 (분류·추출에 적합)
hot = init_chat_model(MODEL, temperature=1.2)   # 다양한 표현 (브레인스토밍에 적합)

prompt = "회의 시작을 알리는 한 문장 인사말을 만들어줘."

print("[temperature=0.0] (반복해도 비슷)")
for _ in range(2):
    print("  -", cold.invoke(prompt).content)

print("[temperature=1.2] (호출마다 달라짐)")
for _ in range(2):
    print("  -", hot.invoke(prompt).content)

# max_tokens로 응답 길이 상한을 둘 수도 있습니다 (비용·지연 통제).
short = init_chat_model(MODEL, max_tokens=20)
print("[max_tokens=20]", short.invoke("LangChain을 설명해줘").content)

> **체크포인트**: 같은 프롬프트인데 temperature가 높을수록 표현 편차가 커지면 파라미터 효과를 이해한 것입니다.

### ④ 스트리밍 (stream으로 토큰 단위 출력)

`invoke`는 답변이 다 완성된 뒤 한 번에 돌려줍니다. `stream`은 토큰이 생성되는 즉시 조각(chunk)으로 흘려보내 체감 속도를 높입니다.

In [ ]:
print("[streaming] ", end="", flush=True)
for chunk in model.stream("LangChain을 두 문장으로 소개해줘."):
    # 각 chunk도 메시지 객체이며 .content에 이번에 생성된 부분 텍스트가 들어 있습니다.
    print(chunk.content, end="", flush=True)
print()

> **체크포인트**: 답변이 한 번에 뜨지 않고 글자가 흘러나오듯 출력되면 스트리밍이 동작하는 것입니다.

### ⑤ 벤더 전환 (openai → google-genai 같은 코드, 키 있을 때만)

핵심은 **모델을 만드는 문자열만 바꾸면 나머지 코드는 그대로**라는 점입니다. `GOOGLE_API_KEY`가 있을 때만 실행됩니다.

In [ ]:
if os.getenv("GOOGLE_API_KEY"):
    gemini = init_chat_model("google-genai:gemini-2.5-flash")  # 클래스 import 변경 없이 문자열만 교체
    resp = gemini.invoke([
        SystemMessage("너는 친절한 한국어 비서다."),
        HumanMessage("LangChain을 한 문장으로 설명해줘."),
    ])
    print("[gemini]", resp.content)
else:
    print("[skip] GOOGLE_API_KEY가 없어 Gemini 전환 예제를 건너뜁니다.")
    print("       MODEL 문자열을 'google-genai:gemini-2.5-flash'로 바꾸면 동일하게 동작합니다.")

> **체크포인트**: OpenAI 예제와 같은 `invoke` 코드가 Gemini에서도 그대로 동작하면 벤더 독립성을 확인한 것입니다.

### ⑥ ChatPromptTemplate + LCEL 체인 재사용

자리표시자(`{역할}`, `{질문}`)를 둔 프롬프트 틀을 만들고, 파이프(`|`)로 모델과 이어 LCEL 체인을 구성합니다. 틀은 그대로 두고 입력만 바꿔 재사용합니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 {역할}이다. 쉽게 설명한다."),
    ("human", "{질문}"),
])

# 파이프(|)로 prompt와 model을 레고 블록처럼 잇습니다.
chain = prompt | model

r1 = chain.invoke({"역할": "교사", "질문": "LCEL을 한 문장으로 설명해줘"})
print("[교사]", r1.content)

r2 = chain.invoke({"역할": "면접관", "질문": "LCEL을 한 문장으로 설명해줘"})
print("[면접관]", r2.content)  # 역할만 바꿨는데 답변 톤이 달라집니다

> **체크포인트**: 변수 값만 바꿔 호출했을 때 응답이 그에 맞게 달라지면 체인이 재사용 가능한 것입니다.

### ⑦ 중괄호 이스케이프 함정 (실패 → 정상)

`ChatPromptTemplate`은 `{ }` 안을 변수로 해석합니다. 본문에 리터럴 중괄호(JSON 예시 등)를 넣으면 변수로 오인해 오류가 납니다. 리터럴 중괄호는 두 번 써서(`{{ }}`) 이스케이프합니다.

In [ ]:
# (1) 실패 예시: {"status":"ok"}의 중괄호를 변수로 보고 KeyError가 발생합니다.
try:
    bad = ChatPromptTemplate.from_messages([
        ("human", '결과를 {"status":"ok"} 형태로 답해라'),
    ])
    bad.invoke({})  # 채워 넣을 값이 없으니 KeyError
    print("[bad] (오류가 나야 정상인데 통과했습니다)")
except KeyError as e:
    print("[bad] 예상대로 KeyError 발생:", e)  # 예: KeyError: '"status"'

# (2) 정상 예시: 리터럴 중괄호는 {{ }}로 이스케이프합니다.
good = ChatPromptTemplate.from_messages([
    ("human", '결과를 {{"status":"ok"}} 형태의 JSON으로 답해라. 질문: {질문}'),
])
msg = good.invoke({"질문": "지금 상태는?"})  # {질문}만 변수로, {{ }}는 리터럴로 처리됩니다
print("[good] 생성된 메시지:", msg.messages[0].content)

> **체크포인트**: (1)에서 `KeyError`를 보고 (2)가 오류 없이 메시지를 만들면 이스케이프 규칙을 이해한 것입니다.

---
## 심화 예제

### ⑧ 구조화 출력 기본 (Pydantic)

원하는 데이터의 형태를 Pydantic 모델로 정의하고, `with_structured_output`으로 모델이 그 스키마에 맞춰 응답하도록 강제합니다. 반환값이 곧 Pydantic 객체이므로 문자열 파싱이 필요 없습니다.

In [ ]:
from pydantic import BaseModel, Field

class Person(BaseModel):
    name: str = Field(description="사람의 이름")
    age: int = Field(description="만 나이, 정수")

structured_model = model.with_structured_output(Person)
person = structured_model.invoke("홍길동은 30살이다")

print("[name/age]", person.name, person.age)  # 예: 홍길동 30
print("[age type]", type(person.age))          # 예: <class 'int'>  (문자열이 아니라 정수)

> **체크포인트**: `person.age`가 정수 타입으로 나오면 구조화 출력에 성공한 것입니다.

### ⑨ include_raw=True 로 원본+파싱 동시 받기

`include_raw=True`를 주면 파싱 결과뿐 아니라 원본 `AIMessage`도 함께 받습니다. 토큰 사용량 확인, 파싱 실패 디버깅, 로깅에 유용합니다. 반환값은 `raw`/`parsed`/`parsing_error` 세 키를 가진 dict입니다.

In [ ]:
class Sentiment(BaseModel):
    label: str = Field(description="positive, negative, neutral 중 하나")
    score: float = Field(description="0.0~1.0 사이의 확신도")

model_with_raw = model.with_structured_output(Sentiment, include_raw=True)
result = model_with_raw.invoke("이번 업데이트 정말 마음에 들어요!")

print("[raw]   ", result["raw"])           # 원본 AIMessage (usage_metadata 등 포함)
print("[parsed]", result["parsed"])         # Sentiment 객체 (검증 통과 시)
print("[error] ", result["parsing_error"])  # 파싱 실패 시 예외, 성공 시 None

parsed = result["parsed"]
print("[label/score]", parsed.label, parsed.score)

> **체크포인트**: `parsed`에는 `Sentiment` 객체가, `raw`에는 원본 메시지가 동시에 담기면 성공입니다.

### ⑩ Optional·중첩 스키마 + 검증 실패 예외 처리

`Optional`과 기본값을 두면 정보가 없을 때 모델이 값을 지어내지 않고 비웁니다. 한 모델이 다른 모델을 필드로 가지는 중첩 스키마도 가능합니다. 응답이 스키마를 못 맞추면 검증 예외가 나므로 호출을 예외 처리로 감쌉니다.

In [ ]:
from typing import Optional, List

class Address(BaseModel):
    city: str = Field(description="도시 이름")
    country: str = Field(description="국가 이름")

class PersonDetail(BaseModel):
    name: str = Field(description="사람의 이름")
    age: Optional[int] = Field(default=None, description="만 나이, 모르면 비워 둔다")
    skills: List[str] = Field(default_factory=list, description="보유 기술 목록")
    address: Optional[Address] = Field(default=None, description="거주지, 모르면 비워 둔다")

detail_model = model.with_structured_output(PersonDetail)

# 정보가 풍부한 입력
try:
    p = detail_model.invoke("김철수는 28살이고 파이썬과 자바를 다룬다. 서울(대한민국)에 산다.")
    print("[full]", p.name, p.age, p.skills, p.address)
except Exception as e:
    print("[full] 구조화 실패, 입력을 보강해 재시도하십시오:", e)

# 정보가 부족한 입력 - 없는 값은 None/빈 리스트로 남습니다.
try:
    p2 = detail_model.invoke("내 이름은 앤디야")
    print("[sparse]", p2.name, p2.age, p2.skills, p2.address)  # 예: 앤디 None [] None
except Exception as e:
    print("[sparse] 구조화 실패:", e)

> **체크포인트**: 부족한 입력에서 `age=None`, `skills=[]`, `address=None`이 나오면 모델이 빈 값을 지어내지 않고 스키마가 안전하게 동작하는 것입니다.

---
다음 단계: 이 챕터를 챗 UI로 옮긴 `streamlit/app_chat.py`도 함께 살펴보십시오.